<a href="https://colab.research.google.com/github/flahbocchino/cardioia-fase5-assistente-paciente/blob/main/ir_alem_1_extracao_clinica_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CardioIA — Fase 5 — Ir Além 1
## IA Generativa para Extração de Informações Clínicas

**Objetivo:** usar um modelo de linguagem (Gemini) para ler anotações clínicas
simuladas, escritas em texto livre (como um médico ou paciente escreveria),
e transformar essa informação em um **JSON estruturado** — o formato que
sistemas de saúde digitais (como o CardioIA) precisam para processar dados.

**Fluxo:** `Texto clínico não estruturado → Prompt estruturado → Gemini API → JSON validado`

**Este notebook é definitivo e autocontido.** Não precisa editar nenhuma célula —
apenas `Ambiente de execução → Executar tudo`.

**Nota sobre o limite de uso (429):** o plano gratuito do Gemini tem um limite
de requisições por dia (RPD) que reseta a cada 24h, e um limite por minuto (RPM).
Este notebook já foi construído para respeitar os dois: ele espera alguns
segundos entre cada chamada e tenta de novo automaticamente (com espera
crescente) se a API responder ocupada.

## 1. Instalação das dependências

In [1]:
!pip install -q -U google-generativeai
print("Dependências instaladas.")


Dependências instaladas.


## 2. Configuração da API

A chave fica guardada no secret `GEMINI_API_KEY` do Colab (ícone de chave 🔑
na lateral esquerda). **Antes de rodar esta célula, confirme dois pontos:**

1. O toggle "Notebook access" do secret `GEMINI_API_KEY` está **ligado**.
2. A chave foi gerada no **mesmo projeto do Google Cloud** que você vai
   consultar depois em https://aistudio.google.com/usage — se checar o
   uso no projeto errado, os números de limite mostrados não vão bater
   com a chave que está sendo usada aqui.

Usamos o alias `gemini-flash-latest`, que o Google sempre mantém apontando
para a versão estável mais recente do Flash — evita que o notebook quebre
quando um modelo específico (ex.: `gemini-1.5-flash`) é descontinuado.

In [2]:
from google.colab import userdata
import google.generativeai as genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

MODELO_NOME = "gemini-flash-latest"
modelo = genai.GenerativeModel(MODELO_NOME)
print(f"Modelo configurado: {MODELO_NOME}")


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Modelo configurado: gemini-flash-latest


## 3. Função de chamada com retry e espera entre requisições

Esta é a parte que resolve o erro 429 de ontem: a cada chamada, o código
espera alguns segundos (respeitando o RPM) e, se ainda assim vier "ocupado"
(429 ou 503), tenta de novo com espera crescente (backoff exponencial),
até 5 tentativas.

In [3]:
import time
import json as _json
import re

def chamar_gemini_com_retry(prompt, max_tentativas=5, espera_inicial=5):
    """Chama o Gemini com backoff exponencial. Levanta erro só se todas as tentativas falharem."""
    espera = espera_inicial
    ultimo_erro = None
    for tentativa in range(1, max_tentativas + 1):
        try:
            resposta = modelo.generate_content(prompt)
            return resposta.text
        except Exception as e:
            ultimo_erro = e
            msg = str(e)
            if "429" in msg or "503" in msg or "RESOURCE_EXHAUSTED" in msg:
                print(f"  (tentativa {tentativa}/{max_tentativas} falhou — {msg[:80]}... aguardando {espera}s)")
                time.sleep(espera)
                espera *= 2
            else:
                # Erro que não é de limite/disponibilidade: não adianta tentar de novo
                raise
    raise RuntimeError(f"Falhou após {max_tentativas} tentativas. Último erro: {ultimo_erro}")

def extrair_json(texto_resposta):
    """Extrai o bloco JSON da resposta do modelo, mesmo se vier com ```json ao redor."""
    match = re.search(r"\{.*\}", texto_resposta, re.DOTALL)
    if not match:
        raise ValueError(f"Não encontrei JSON na resposta: {texto_resposta[:200]}")
    return _json.loads(match.group(0))

print("Funções de apoio prontas.")


Funções de apoio prontas.


## 4. Textos clínicos simulados (entrada não estruturada)

In [4]:
textos_clinicos_simulados = [
    """
    Paciente relata dor no peito de início súbito há cerca de 40 minutos,
    caracterizada como aperto, irradiando para o braço esquerdo. Refere
    também sudorese e falta de ar. Pressão arterial aferida: 158/98 mmHg.
    Frequência cardíaca: 110 bpm. Paciente é tabagista há 15 anos (cerca de
    1 maço/dia) e faz uso contínuo de Losartana 50mg. Nega uso de outras
    medicações.
    """,
    """
    Paciente do sexo feminino, 34 anos, comparece à consulta de rotina.
    Nega dor torácica ou falta de ar. Relata palpitações ocasionais,
    principalmente após consumo de café. Pressão arterial: 112/74 mmHg.
    Frequência cardíaca: 78 bpm. Não fumante. Histórico familiar de
    hipertensão (mãe). Nenhuma medicação em uso.
    """,
    """
    Paciente masculino, 58 anos, ex-tabagista (parou há 3 anos, fumou por
    20 anos). Relata falta de ar progressiva aos esforços nas últimas 2
    semanas, e episódios de tontura. Pressão arterial: 145/92 mmHg.
    Frequência cardíaca: 95 bpm. Colesterol total (último exame): 240 mg/dL.
    Diabético tipo 2, em uso de Metformina 850mg 2x/dia.
    """,
]

print(f"{len(textos_clinicos_simulados)} textos clínicos carregados.")


3 textos clínicos carregados.


## 5. Prompt estruturado

O prompt instrui o Gemini a devolver **apenas JSON**, com um schema fixo —
isso é o que torna a saída confiável o bastante para alimentar outro sistema
(como o CardioIA) sem depender de interpretação manual.

In [5]:
PROMPT_BASE = """Você é um assistente de extração de dados clínicos. Leia o texto
abaixo, escrito em linguagem livre por um profissional de saúde, e extraia
as informações em um JSON com EXATAMENTE este formato (use null quando a
informação não aparecer no texto):

{{
  "idade": <número ou null>,
  "sexo": "<masculino, feminino ou null>",
  "sintomas": ["<lista de sintomas relatados>"],
  "tempo_sintomas": "<duração relatada, ex: '40 minutos', '2 semanas', ou null>",
  "sinais_vitais": {{
    "pressao_arterial": "<ex: 158/98 mmHg, ou null>",
    "frequencia_cardiaca_bpm": <número ou null>
  }},
  "tabagismo": {{
    "status": "<tabagista, ex-tabagista, nao_fumante ou null>",
    "detalhes": "<ex: 15 anos, 1 maço/dia, ou null>"
  }},
  "comorbidades": ["<lista, ex: diabetes, hipertensao>"],
  "medicacoes_em_uso": ["<lista de medicações e dose, se citada>"],
  "historico_familiar": "<resumo, ou null>",
  "nivel_urgencia_sugerido": "<baixo, medio ou alto, com base nos sintomas e sinais descritos>"
}}

Responda APENAS com o JSON, sem nenhum texto antes ou depois, sem markdown.

Texto clínico:
\"\"\"{texto}\"\"\"
"""

print("Prompt definido.")


Prompt definido.


## 6. Execução — extração dos 3 textos

Espera de ~8 segundos entre cada chamada para não bater no limite por minuto
(RPM), além do retry automático em caso de 429/503.

In [6]:
resultados = []

for i, texto in enumerate(textos_clinicos_simulados, start=1):
    print(f"--- Processando texto {i}/{len(textos_clinicos_simulados)} ---")
    prompt = PROMPT_BASE.format(texto=texto.strip())
    resposta_bruta = chamar_gemini_com_retry(prompt)
    dados_extraidos = extrair_json(resposta_bruta)
    resultados.append({"texto_original": texto.strip(), "dados_extraidos": dados_extraidos})
    nivel = dados_extraidos.get("nivel_urgencia_sugerido")
    print(f"  OK — nível de urgência sugerido: {nivel}")
    if i < len(textos_clinicos_simulados):
        print("  Aguardando 8s antes do próximo texto...")
        time.sleep(8)

print()
print("Extração concluída para todos os textos.")


--- Processando texto 1/3 ---


ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 57465.88ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 41164.99ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 33991.67ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 71873.51ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 26773.83ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 12493.62ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4486.18ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateCont

  OK — nível de urgência sugerido: alto
  Aguardando 8s antes do próximo texto...
--- Processando texto 2/3 ---
  OK — nível de urgência sugerido: baixo
  Aguardando 8s antes do próximo texto...
--- Processando texto 3/3 ---


ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4156.41ms


  OK — nível de urgência sugerido: medio

Extração concluída para todos os textos.


## 7. Resultado formatado (validação visual)

In [7]:
for i, r in enumerate(resultados, start=1):
    print(f"===== Texto {i} =====")
    print(r["texto_original"][:120].replace(chr(10), " ") + "...")
    print("--- JSON extraído ---")
    print(_json.dumps(r["dados_extraidos"], indent=2, ensure_ascii=False))
    print()


===== Texto 1 =====
Paciente relata dor no peito de início súbito há cerca de 40 minutos,     caracterizada como aperto, irradiando para o b...
--- JSON extraído ---
{
  "idade": null,
  "sexo": null,
  "sintomas": [
    "dor no peito em aperto com irradiação para o braço esquerdo",
    "sudorese",
    "falta de ar"
  ],
  "tempo_sintomas": "cerca de 40 minutos",
  "sinais_vitais": {
    "pressao_arterial": "158/98 mmHg",
    "frequencia_cardiaca_bpm": 110
  },
  "tabagismo": {
    "status": "tabagista",
    "detalhes": "há 15 anos, cerca de 1 maço/dia"
  },
  "comorbidades": [],
  "medicacoes_em_uso": [
    "Losartana 50mg"
  ],
  "historico_familiar": null,
  "nivel_urgencia_sugerido": "alto"
}

===== Texto 2 =====
Paciente do sexo feminino, 34 anos, comparece à consulta de rotina.     Nega dor torácica ou falta de ar. Relata palpita...
--- JSON extraído ---
{
  "idade": 34,
  "sexo": "feminino",
  "sintomas": [
    "palpitações ocasionais"
  ],
  "tempo_sintomas": null,
  "sinais_vi

## 8. Salvando o resultado (evidência para entrega)

Gera um arquivo `resultados_extracao_clinica.json` com todas as extrações —
esse arquivo vai para o repositório GitHub como evidência de que o Ir Além 1
funcionou de ponta a ponta.

In [8]:
with open("resultados_extracao_clinica.json", "w", encoding="utf-8") as f:
    _json.dump(resultados, f, indent=2, ensure_ascii=False)

print("Arquivo salvo: resultados_extracao_clinica.json")

from google.colab import files
files.download("resultados_extracao_clinica.json")


Arquivo salvo: resultados_extracao_clinica.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>